In [3]:
import pandas as pd
import numpy as np
import io
import requests

BASE_URL = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    "?parameters=T2M&community=RE"
    "&longitude={lon}&latitude={lat}"
    "&start={start}&end={end}&format=CSV"
)

# Capitais das duas regioes que cobriremos.
# Sudeste:      SP (Sao Paulo), RJ (Rio de Janeiro), MG (Belo Horizonte), ES (Vitoria)
# Centro-Oeste: DF (Brasilia),  GO (Goiania),        MT (Cuiaba),         MS (Campo Grande)
LOCAIS = {
    # Sudeste
    "SP": {"lat": -23.5505, "lon": -46.6333},
    "RJ": {"lat": -22.9068, "lon": -43.1729},
    "MG": {"lat": -19.9167, "lon": -43.9378},
    "ES": {"lat": -20.3155, "lon": -40.3378},
    # Centro-Oeste
    "DF": {"lat": -15.7942, "lon": -47.8822},
    "GO": {"lat": -16.6864, "lon": -49.2643},
    "MT": {"lat": -15.6010, "lon": -56.0974},
    "MS": {"lat": -20.4486, "lon": -54.6295},
}

# Periodos a buscar: cada tupla e (start, end) no formato YYYYMMDD.
PERIODOS = [
    ("20250101", "20251231"),
    ("20260101", "20261231"),
]

def parse_nasa_daily_csv(text, uf):
    lines = text.splitlines()
    header_idx = next(
        (i for i, line in enumerate(lines) if line.startswith("YEAR")),
        None
    )
    if header_idx is None:
        return None
    df = pd.read_csv(io.StringIO("\n".join(lines[header_idx:])))
    df = df.rename(columns={"MO": "mes", "DY": "dia", "T2M": "temperatura"})
    # NASA POWER usa -999 como marcador de dado faltante. np.nan mantem o dtype float.
    df["temperatura"] = df["temperatura"].replace(-999, np.nan)
    df["data"] = (
        df["YEAR"].astype(str) + "-"
        + df["mes"].astype(str).str.zfill(2) + "-"
        + df["dia"].astype(str).str.zfill(2)
    )
    df["uf"] = uf
    return df[["uf", "data", "temperatura"]]

dfs = []

for uf, coords in LOCAIS.items():
    for start, end in PERIODOS:
        url = BASE_URL.format(
            lat=coords["lat"], lon=coords["lon"],
            start=start, end=end
        )
        response = requests.get(url)
        if response.status_code == 422:
            print(f"Dados {start[:4]} nao disponiveis para {uf} - pulando.")
            continue
        response.raise_for_status()
        df = parse_nasa_daily_csv(response.text, uf)
        if df is not None:
            dfs.append(df)

if not dfs:
    raise RuntimeError("Nenhum dado foi obtido da NASA POWER.")

dados = pd.concat(dfs, ignore_index=True)

# Media diaria simples entre as 8 capitais (Sudeste + Centro-Oeste).
se_co = (
    dados
    .groupby("data", as_index=False)["temperatura"]
    .mean()
    .rename(columns={"temperatura": "temperatura_media"})
)

se_co["temperatura_media"] = se_co["temperatura_media"].round(2)

se_co.to_csv("dataset/temperatura_diaria_sudeste_centro_oeste.csv", index=False)

print(f"{len(se_co)} dias gerados.")
print(se_co.head())
print(se_co.tail())

510 dias gerados.
         data  temperatura_media
0  2025-01-01              24.77
1  2025-01-02              24.60
2  2025-01-03              24.97
3  2025-01-04              24.80
4  2025-01-05              24.62
           data  temperatura_media
505  2026-05-21              20.73
506  2026-05-22              20.02
507  2026-05-23                NaN
508  2026-05-24                NaN
509  2026-05-25                NaN
